In [1]:
const appRoutes = [
  "/",
  "/old",
  "/login",
  "/ui",
  "/contact-us",
  "/settings/:tab?",
  "/search",
  "/blog/:slug?",
];

const routes = [];
routes.push("/resources/:filename+");
routes.push("/build/:filename+");
for (const entry of appRoutes) {
  routes.push(entry);
}

10

In [7]:
const testUrls = ["/", "/login", "/ui", "/settings", "/settings/", "/settings/main", "/resources/style.css", "/build/something.js", "/blog", "/blog/Title"];

const matches = new Map();
testUrls.forEach((incomingUrl) => {
  const pathWithParams = incomingUrl.split("?")[0];
  const routeParams: Record<string, string | null> = {};
  let matchedRoute: string | undefined;
  // First, check for an exact match
  matchedRoute = routes.find((r) => r === pathWithParams);
  if (!matchedRoute) {
    // Now check for a match using parameterized routes
    for (const routePath of routes) {
      // ORIGINAL
      // const regexPath = routePath.replace(/:\w+\??\+?/g, (match) => {
      //   if (match.endsWith("+")) {
      //     return "(.+)";
      //   } else if (match.endsWith("?")) {
      //     return "([^/]*)";
      //   } else {
      //     return "([^/]+)";
      //   }
      // }) + "($|/)";
      // _______________

      const regexPath = routePath.replace(/\/:\w+\??\+?/g, (match) => {
        if (match.endsWith("+")) {
          return "(.+)";
        } else if (match.endsWith("?")) {
          return "([^/]*)";
        } else {
          return "([^/]+)";
        }
      }) + "($|/)";

      // ______________
      // NEW
      // const regexPath = routePath.replace(/\/:\w+\??\+?/g, (match) => {
      //   if (match.endsWith("+")) {
      //     return "/(.+)";
      //   } else if (match.endsWith("?")) {
      //     return "(?:/([^/]+))?";
      //   } else {
      //     return "/([^/]+)";
      //   }
      // }) + "(?:/?)$";
      //
      const match = pathWithParams.match(new RegExp(`^${regexPath}`));
      if (match) {
        const paramNames = (routePath.match(/:\w+\??\+?/g) || []).map((p) =>
          p.substring(1).replace(/[\?\+]/g, "")
        );
        for (let i = 0; i < paramNames.length; i++) {
          routeParams[paramNames[i]] = match[i + 1] || null;
        }
        matchedRoute = routePath;
        break;
      }
    }
  }
  matches.set(pathWithParams, matchedRoute);
});


matches

Map(10) {
  "/" => "/",
  "/login" => "/login",
  "/ui" => "/ui",
  "/settings" => "/settings/:tab?",
  "/settings/" => "/settings/:tab?",
  "/settings/main" => "/settings/:tab?",
  "/resources/style.css" => "/resources/:filename+",
  "/build/something.js" => "/build/:filename+",
  "/blog" => "/blog/:slug?",
  "/blog/Title" => "/blog/:slug?"
}